
# Tech Challenge FIAP — Fase 3
## Fine-tuning de LLM médica — Exploração e Preprocessing

### Objetivo

Este notebook apresenta as etapas iniciais de preparação dos dados para o fine-tuning de um Large Language Model (LLM) especializado em perguntas e respostas médicas.

O projeto faz parte do Tech Challenge da Fase 3 da Pós-Tech de Inteligência Artificial para Devs da FIAP.

### Dataset

Será utilizado o **MedQuAD (Medical Question Answering Dataset)**, disponibilizado pela comunidade no Hugging Face.

**Fonte:** https://huggingface.co/datasets/lavita/MedQuAD

O dataset contém perguntas e respostas relacionadas a condições médicas, diagnósticos, tratamentos e outros temas de saúde.

### Etapas

1. Carregamento e exploração dos dados.
2. Análise da qualidade e identificação de valores ausentes.
3. Limpeza e padronização dos registros.
4. Separação em conjuntos de treinamento, validação e teste.
5. Exportação dos dados preparados para o fine-tuning.

**Observação:** O assistente será destinado ao apoio à decisão clínica e não substituirá a avaliação de profissionais de saúde.



## 1. Configuração do ambiente

O processamento inicial será realizado no Google Colab, utilizando Python e as bibliotecas Hugging Face Datasets e Pandas.

A utilização do Colab permite desenvolver e executar posteriormente o treinamento da LLM em um ambiente com suporte a GPU.


In [26]:
!pip install -q datasets pandas

import pandas as pd
from datasets import load_dataset


## 2. Carregamento do dataset

O MedQuAD será carregado diretamente do Hugging Face por meio da biblioteca `datasets`.

O dataset disponibilizado possui 47.441 registros e 13 atributos, incluindo perguntas, respostas, fontes, categorias e identificadores dos documentos.

Nesta etapa, os registros serão convertidos para um DataFrame Pandas para facilitar a exploração e o processamento.


In [27]:
dataset = load_dataset("lavita/MedQuAD")

df = dataset["train"].to_pandas()

print(f"Registros: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

display(df.head())

Registros: 47441
Colunas: 13


,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...



## 3. Análise exploratória dos dados (EDA)

A análise exploratória tem como objetivo compreender a estrutura do MedQuAD e identificar características que possam impactar o fine-tuning.

Serão avaliados:

- Tipos e nomes dos atributos.
- Quantidade de registros.
- Valores ausentes nas perguntas e respostas.
- Distribuição das fontes de informação.
- Tipos de perguntas disponíveis.

Essas informações ajudarão a definir os critérios de limpeza e seleção dos exemplos utilizados no treinamento.


In [28]:
# Informações gerais
df.info()

# Lista de colunas
print("\nColunas disponíveis:")
print(df.columns.tolist())

# Exemplo de perguntas e respostas
sample = df[["question", "answer"]].head(5).copy()
sample["answer"] = sample["answer"].str.slice(0, 200)

display(sample)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47441 entries, 0 to 47440
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   document_id          47436 non-null  object
 1   document_source      47441 non-null  object
 2   document_url         47441 non-null  object
 3   category             32010 non-null  object
 4   umls_cui             31417 non-null  object
 5   umls_semantic_types  31375 non-null  object
 6   umls_semantic_group  31417 non-null  object
 7   synonyms             24669 non-null  object
 8   question_id          47441 non-null  object
 9   question_focus       47427 non-null  object
 10  question_type        47441 non-null  object
 11  question             47441 non-null  object
 12  answer               16407 non-null  object
dtypes: object(13)
memory usage: 4.7+ MB

Colunas disponíveis:
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_typ

,question,answer
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...



## 4. Análise de valores ausentes

A identificação de dados ausentes é fundamental para garantir a qualidade do fine-tuning.

Como o treinamento supervisionado utilizará pares de perguntas e respostas, registros sem um desses campos não poderão ser utilizados diretamente.

Serão considerados inválidos:

- Perguntas nulas ou vazias.
- Respostas nulas ou vazias.
- Campos contendo apenas espaços em branco.

A quantidade de registros válidos será analisada antes de aplicar qualquer remoção.


In [29]:
# Quantidade de valores nulos por coluna
missing = df.isna().sum().sort_values(ascending=False)

display(
    pd.DataFrame({
        "Valores ausentes": missing,
        "Percentual (%)": (missing / len(df) * 100).round(2)
    })
)

# Verificação específica de perguntas e respostas
valid_questions = df["question"].fillna("").str.strip().ne("")
valid_answers = df["answer"].fillna("").str.strip().ne("")

print(f"Total de registros: {len(df)}")
print(f"Perguntas válidas: {valid_questions.sum()}")
print(f"Respostas válidas: {valid_answers.sum()}")
print(f"Pares válidos: {(valid_questions & valid_answers).sum()}")

,Valores ausentes,Percentual (%)
answer,31034,65.42
synonyms,22772,48.00
umls_semantic_types,16066,33.87
umls_semantic_group,16024,33.78
umls_cui,16024,33.78
category,15431,32.53
question_focus,14,0.03
document_id,5,0.01
document_url,0,0.00
document_source,0,0.00


Total de registros: 47441
Perguntas válidas: 47441
Respostas válidas: 16407
Pares válidos: 16407



## 5. Distribuição das fontes e tipos de perguntas

O MedQuAD reúne informações provenientes de diferentes fontes médicas.

Nesta etapa, analisaremos a distribuição dos registros por fonte e por tipo de pergunta.

Também verificaremos quantos registros de cada fonte possuem respostas válidas.

Essa análise será importante para selecionar exemplos representativos e evitar que o treinamento fique concentrado em apenas um tipo de pergunta.


In [30]:
# Distribuição por fonte
source_distribution = (
    df.assign(valid_answer=valid_answers)
      .groupby("document_source", dropna=False)
      .agg(
          total=("question", "size"),
          valid_answers=("valid_answer", "sum")
      )
      .sort_values("total", ascending=False)
)

display(source_distribution)

# Distribuição por tipo de pergunta
question_types = (
    df.loc[valid_questions & valid_answers, "question_type"]
      .value_counts(dropna=False)
)

display(question_types)

,total,valid_answers
document_source,,
ADAM,17348,0
MPlusDrugs,12889,0
GHR,5430,5430
GARD,5394,5389
NIDDK,1192,1192
NINDS,1088,1088
MPlusHealthTopics,981,981
MPlusHerbsSupplements,792,0
NIHSeniorHealth,769,769


,count
question_type,
information,4535
symptoms,2748
treatment,2442
inheritance,1446
frequency,1120
genetic changes,1087
causes,727
exams and tests,653
research,395



## 6. Preprocessing e limpeza dos dados

Com base na análise exploratória, será realizada a preparação dos registros para o fine-tuning supervisionado.

As seguintes operações serão aplicadas:

1. Remoção de registros sem perguntas ou respostas válidas.
2. Normalização de espaços em branco.
3. Identificação e remoção de pares duplicados.
4. Preservação dos metadados e da origem dos documentos.

As respostas serão preservadas integralmente, sem resumir seu conteúdo nesta etapa.

A limpeza busca melhorar a consistência do dataset sem alterar o significado das informações médicas.


In [31]:
# Criar uma cópia para preservar o dataset original
df_clean = df.copy()

# Substituir valores nulos e remover espaços nas extremidades
for column in ["question", "answer"]:
    df_clean[column] = (
        df_clean[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# Normalizar espaços das perguntas
df_clean["question"] = (
    df_clean["question"]
    .str.replace(r"\s+", " ", regex=True)
)

# Normalizar quebras de linha nas respostas
df_clean["answer"] = (
    df_clean["answer"]
    .str.replace("\r\n", "\n", regex=False)
    .str.replace(r"\n{3,}", "\n\n", regex=True)
)

# Remover registros sem pergunta ou resposta
df_clean = df_clean[
    df_clean["question"].ne("") &
    df_clean["answer"].ne("")
].copy()

print(f"Registros originais: {len(df)}")
print(f"Registros válidos: {len(df_clean)}")
print(f"Registros removidos: {len(df) - len(df_clean)}")

Registros originais: 47441
Registros válidos: 16407
Registros removidos: 31034



## 7. Identificação de registros duplicados

Registros duplicados podem gerar redundância durante o treinamento e influenciar a avaliação do modelo.

Serão consideradas duplicatas as ocorrências com a mesma pergunta e a mesma resposta, desconsiderando diferenças de maiúsculas, minúsculas e espaços em branco.

Perguntas iguais acompanhadas de respostas diferentes não serão removidas automaticamente, pois podem representar contextos médicos distintos.

Os registros serão analisados antes de qualquer redução adicional do dataset.


In [32]:
# Normalizar textos apenas para comparação
normalized_questions = (
    df_clean["question"]
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

normalized_answers = (
    df_clean["answer"]
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

# Identificar pares duplicados
duplicate_keys = pd.DataFrame({
    "question": normalized_questions,
    "answer": normalized_answers
})

duplicate_mask = duplicate_keys.duplicated()

print(f"Pares duplicados: {duplicate_mask.sum()}")

# Remover duplicatas
df_clean = (
    df_clean.loc[~duplicate_mask]
    .reset_index(drop=True)
)

print(f"Registros após limpeza: {len(df_clean)}")

Pares duplicados: 48
Registros após limpeza: 16359



## 8. Análise do comprimento das perguntas e respostas

O comprimento dos textos influencia os requisitos computacionais do fine-tuning.

Nesta etapa, serão calculadas estatísticas sobre a quantidade de palavras presentes nas perguntas e respostas.

A análise permitirá identificar respostas excepcionalmente longas e orientar a futura definição do tamanho máximo das sequências de treinamento.

Nenhuma resposta será truncada automaticamente nesta etapa, evitando a perda de informações médicas relevantes.


In [33]:
# Calcular comprimento em palavras
df_clean["question_words"] = (
    df_clean["question"].str.split().str.len()
)

df_clean["answer_words"] = (
    df_clean["answer"].str.split().str.len()
)

# Estatísticas descritivas
display(
    df_clean[
        ["question_words", "answer_words"]
    ].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]).round(2)
)

# Exemplos de respostas mais longas
display(
    df_clean.nlargest(5, "answer_words")[
        [
            "document_source",
            "question_type",
            "question",
            "answer_words"
        ]
    ]
)

,question_words,answer_words
count,16359.00,16359.00
mean,8.21,200.37
std,2.38,246.87
min,3.00,1.00
50%,8.00,138.00
75%,10.00,251.00
90%,11.00,378.00
95%,12.00,517.10
99%,15.00,1263.26
max,27.00,4281.00


,document_source,question_type,question,answer_words
12602,CancerGov,treatment,What are the treatments for Breast Cancer ?,4281
12416,CancerGov,treatment,What are the treatments for Childhood Soft Tis...,3708
12472,CancerGov,information,What is (are) Childhood Vascular Tumors ?,3692
12764,NHLBI,treatment,What are the treatments for Sickle Cell Disease ?,3594
12471,CancerGov,treatment,What are the treatments for Childhood Rhabdomy...,3461



## 9. Análise de respostas curtas e longas

A análise estatística identificou grande variação no tamanho das respostas, desde registros com apenas uma palavra até respostas com milhares de palavras.

Antes de definir os limites de sequência para o fine-tuning, serão inspecionados os exemplos extremos.

**Objetivos:**

- Identificar respostas muito curtas que possam estar incompletas.
- Verificar a quantidade de respostas excessivamente longas.
- Analisar as fontes associadas a esses registros.
- Evitar a remoção ou truncamento de informações médicas relevantes.

Os limites utilizados nesta análise são exploratórios e não representam critérios definitivos de exclusão.


In [34]:
# Análise de respostas muito curtas
short_answers = df_clean[df_clean["answer_words"] <= 10]

print(f"Respostas com até 10 palavras: {len(short_answers)}")

display(
    short_answers[
        ["document_source", "question", "answer", "answer_words"]
    ].head(10)
)

# Análise de respostas longas
print("\nDistribuição de respostas longas:")

for threshold in [512, 800, 1000, 2000]:
    count = (df_clean["answer_words"] > threshold).sum()
    percentage = count / len(df_clean) * 100

    print(f"Mais de {threshold} palavras: {count} ({percentage:.2f}%)")

# Fontes das respostas com mais de 1000 palavras
long_answers = df_clean[df_clean["answer_words"] > 1000]

display(
    long_answers["document_source"]
    .value_counts()
    .rename_axis("Fonte")
    .reset_index(name="Quantidade")
)

Respostas com até 10 palavras: 120


,document_source,question,answer,answer_words
61,GHR,How many people are affected by Pallister-Hall...,This condition is very rare; its prevalence is...,9
81,GHR,How many people are affected by primary ciliar...,Primary ciliary dyskinesia occurs in approxima...,10
141,GHR,How many people are affected by Greig cephalop...,This condition is very rare; its prevalence is...,9
231,GHR,How many people are affected by CATSPER1-relat...,The prevalence of CATSPER1-related nonsyndromi...,9
271,GHR,How many people are affected by Adams-Oliver s...,Adams-Oliver syndrome is a rare disorder; its ...,10
401,GHR,How many people are affected by ovarian cancer ?,"Ovarian cancer affects about 12 in 100,000 wom...",10
426,GHR,How many people are affected by medullary cyst...,"MCKD1 is a rare disorder, although its prevale...",10
516,GHR,How many people are affected by familial gluco...,The prevalence of familial glucocorticoid defi...,8
541,GHR,How many people are affected by childhood myoc...,The prevalence of childhood myocerebrohepatopa...,8
816,GHR,How many people are affected by X-linked agamm...,"XLA occurs in approximately 1 in 200,000 newbo...",8



Distribuição de respostas longas:
Mais de 512 palavras: 837 (5.12%)
Mais de 800 palavras: 353 (2.16%)
Mais de 1000 palavras: 244 (1.49%)
Mais de 2000 palavras: 56 (0.34%)


,Fonte,Quantidade
0,CancerGov,141
1,NHLBI,38
2,NIDDK,34
3,NIHSeniorHealth,23
4,CDC,6
5,GARD,2


In [35]:
# Remover respostas extremamente curtas identificadas
# como inválidas durante a inspeção manual

invalid_mask = (
    (df_clean["answer_words"] <= 3) &
    (df_clean["answer"].str.casefold() == "topics")
)

print(f"Registros inválidos: {invalid_mask.sum()}")

df_clean = (
    df_clean.loc[~invalid_mask]
    .reset_index(drop=True)
)

print(f"Registros após revisão: {len(df_clean)}")

Registros inválidos: 1
Registros após revisão: 16358



## 10. Separação dos dados em treinamento, validação e teste

Os dados serão divididos em três conjuntos, com proporções aproximadas de 80% para treinamento, 10% para validação e 10% para teste.

Durante a validação inicial, foram identificados documentos compartilhados entre os conjuntos, apesar da separação por `question_focus`.

Para reduzir esse risco de vazamento de dados, a estratégia foi aprimorada utilizando agrupamento por componentes conectados.

Registros que compartilham o mesmo documento ou assunto médico serão agrupados, incluindo conexões indiretas entre registros.

A divisão será realizada utilizando `GroupShuffleSplit`, com semente aleatória fixa para garantir reprodutibilidade.

Essa estratégia reduz o risco de vazamento estrutural, embora não elimine completamente a possibilidade de conteúdos semanticamente semelhantes entre os conjuntos.


In [36]:
from sklearn.model_selection import GroupShuffleSplit

df_split = df_clean.copy().reset_index(drop=True)

# Union-Find para agrupar registros relacionados
parent = list(range(len(df_split)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


# Identificar registros relacionados
group_owner = {}

for idx, row in df_split.iterrows():
    keys = []

    # Agrupar pelo assunto médico
    focus = row["question_focus"]

    if pd.notna(focus) and str(focus).strip():
        normalized_focus = " ".join(
            str(focus).casefold().split()
        )
        keys.append(("focus", normalized_focus))

    # Agrupar pelo documento de origem
    document_id = row["document_id"]

    if pd.notna(document_id) and str(document_id).strip():
        keys.append((
            "document",
            str(row["document_source"]),
            str(document_id).strip()
        ))

    for key in keys:
        if key in group_owner:
            union(idx, group_owner[key])
        else:
            group_owner[key] = idx


# Atribuir grupo final
df_split["split_group"] = [
    find(i) for i in range(len(df_split))
]

group_sizes = df_split["split_group"].value_counts()

print(f"Total de registros: {len(df_split)}")
print(f"Total de grupos: {len(group_sizes)}")
print(f"Maior grupo: {group_sizes.max()} registros")

# Separação treino / temporário
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    splitter.split(
        df_split,
        groups=df_split["split_group"]
    )
)

train_df = df_split.iloc[train_idx].copy()
temp_df = df_split.iloc[temp_idx].copy()

# Separação validação / teste
splitter_temp = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    splitter_temp.split(
        temp_df,
        groups=temp_df["split_group"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Resultados
for name, subset in [
    ("Treinamento", train_df),
    ("Validação", val_df),
    ("Teste", test_df)
]:
    percentage = len(subset) / len(df_split) * 100
    print(f"{name}: {len(subset)} ({percentage:.2f}%)")

# Validar grupos
train_groups = set(train_df["split_group"])
val_groups = set(val_df["split_group"])
test_groups = set(test_df["split_group"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("\nDivisão validada: nenhum grupo compartilhado.")

Total de registros: 16358
Total de grupos: 4734
Maior grupo: 58 registros
Treinamento: 13089 (80.02%)
Validação: 1632 (9.98%)
Teste: 1637 (10.01%)

Divisão validada: nenhum grupo compartilhado.



## 11. Validação dos conjuntos de dados

Após a divisão dos registros, serão realizadas verificações adicionais para identificar possíveis problemas que comprometam a avaliação do modelo.

Serão avaliados:

- Distribuição das fontes entre treinamento, validação e teste.
- Distribuição dos tipos de perguntas.
- Possível compartilhamento de documentos entre os conjuntos.
- Respostas extremamente curtas que possam indicar registros incompletos.

Essas verificações complementam a separação por `question_focus` e ajudam a reduzir riscos de vazamento de dados e desequilíbrio na avaliação.


In [37]:
# Reunir os conjuntos para análise
df_analysis = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test")
    ],
    ignore_index=True
)

# Distribuição percentual das fontes
print("Distribuição por fonte (%):")

source_distribution = pd.crosstab(
    df_analysis["split"],
    df_analysis["document_source"],
    normalize="index"
) * 100

display(source_distribution.round(2))

# Distribuição percentual dos tipos de perguntas
print("\nDistribuição por tipo de pergunta (%):")

type_distribution = pd.crosstab(
    df_analysis["split"],
    df_analysis["question_type"],
    normalize="index"
) * 100

display(type_distribution.round(2))

Distribuição por fonte (%):


document_source,CDC,CancerGov,GARD,GHR,MPlusHealthTopics,NHLBI,NIDDK,NIHSeniorHealth,NINDS
split,,,,,,,,,
test,0.67,5.99,31.22,36.04,6.35,3.05,6.72,4.83,5.13
train,1.86,4.46,33.24,33.16,5.72,3.38,6.77,4.78,6.63
validation,0.92,2.88,32.29,30.64,7.84,4.04,9.07,3.98,8.33



Distribuição por tipo de pergunta (%):


question_type,causes,complications,considerations,exams and tests,frequency,genetic changes,information,inheritance,outlook,prevention,research,stages,support groups,susceptibility,symptoms,treatment
split,,,,,,,,,,,,,,,,
test,3.91,0.24,1.34,3.42,7.45,7.21,27.92,9.22,2.02,1.22,2.20,0.61,0.00,2.32,16.31,14.60
train,4.34,0.30,1.34,4.00,6.85,6.64,27.47,8.79,2.20,1.31,2.41,0.47,0.01,1.96,17.01,14.92
validation,4.66,0.18,1.90,4.35,6.25,6.13,28.62,8.88,2.45,0.98,2.70,0.37,0.00,1.84,15.50,15.20


In [38]:
def get_document_keys(dataframe):
    valid = dataframe["document_id"].notna()

    documents = dataframe.loc[valid].copy()

    documents = documents[
        documents["document_id"].astype(str).str.strip().ne("")
    ]

    keys = (
        documents["document_source"].astype(str)
        + "::"
        + documents["document_id"].astype(str).str.strip()
    )

    return set(keys)


train_docs = get_document_keys(train_df)
val_docs = get_document_keys(val_df)
test_docs = get_document_keys(test_df)

shared_train_val = train_docs & val_docs
shared_train_test = train_docs & test_docs
shared_val_test = val_docs & test_docs

print(f"Documentos treino/validação: {len(shared_train_val)}")
print(f"Documentos treino/teste: {len(shared_train_test)}")
print(f"Documentos validação/teste: {len(shared_val_test)}")

if shared_train_val or shared_train_test or shared_val_test:
    print("\nAtenção: existem documentos compartilhados.")
    print("Precisaremos revisar a divisão antes do treinamento.")
else:
    print("\nNenhum documento compartilhado.")

Documentos treino/validação: 0
Documentos treino/teste: 0
Documentos validação/teste: 0

Nenhum documento compartilhado.


In [39]:
very_short_answers = df_clean[
    df_clean["answer_words"] <= 3
]

print(
    f"Respostas com até 3 palavras: "
    f"{len(very_short_answers)}"
)

display(
    very_short_answers[
        [
            "document_source",
            "question",
            "answer",
            "answer_words"
        ]
    ]
)

Respostas com até 3 palavras: 0


,document_source,question,answer,answer_words



## 12. Exportação dos dados

Após a limpeza, divisão e validação, os conjuntos serão exportados no formato JSON Lines (JSONL).

Serão gerados três arquivos:

- `train.jsonl`: dados para o fine-tuning.
- `validation.jsonl`: dados para acompanhamento e ajuste do treinamento.
- `test.jsonl`: dados reservados para avaliação final.

Os arquivos preservarão perguntas, respostas e metadados de origem, permitindo rastrear os exemplos e reproduzir os experimentos.

Os arquivos serão armazenados no Google Drive para evitar perda de dados quando a sessão do Google Colab for encerrada.


In [41]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

output_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Validar total de registros
total = len(train_df) + len(val_df) + len(test_df)

assert total == 16358, f"Total inesperado: {total}"

# Garantir ausência de documentos compartilhados
assert not shared_train_val
assert not shared_train_test
assert not shared_val_test

columns = [
    "question",
    "answer",
    "document_source",
    "document_id",
    "question_type",
    "question_focus",
    "split_group"
]

datasets_to_export = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

for name, dataframe in datasets_to_export.items():
    path = output_dir / f"{name}.jsonl"

    dataframe[columns].to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"{name}: {len(dataframe)} registros")
    print(f"Salvo em: {path}\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: 13089 registros
Salvo em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/train.jsonl

validation: 1632 registros
Salvo em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/validation.jsonl

test: 1637 registros
Salvo em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/processed/test.jsonl




## 13. Considerações finais

O processo de exploração e preparação do MedQuAD permitiu identificar problemas de qualidade, remover registros inadequados e organizar os dados para o fine-tuning supervisionado.

### Resultados

- **Registros originais:** 47.441
- **Registros sem respostas válidas:** 31.034
- **Pares duplicados removidos:** 48
- **Registro inválido identificado manualmente:** 1
- **Dataset final:** 16.358 pares de perguntas e respostas

### Estratégia de preparação

Foram realizadas normalização textual, remoção de duplicatas, inspeção de respostas extremas e separação dos dados em treinamento, validação e teste.

A divisão utilizou componentes conectados por assunto médico e documento de origem, reduzindo o risco de vazamento entre os conjuntos.

As distribuições das fontes e dos principais tipos de perguntas permaneceram relativamente equilibradas.

### Limitações

O MedQuAD contém predominantemente conteúdo médico em inglês e apresenta desequilíbrio entre categorias e fontes.

A ausência de documentos compartilhados não elimina completamente a possibilidade de similaridade semântica entre os conjuntos.

Os dados também representam conhecimento médico público e não substituem protocolos hospitalares internos.

### Próximas etapas

1. Preparar dados hospitalares sintéticos, incluindo protocolos, perguntas frequentes e modelos de documentos clínicos.
2. Escolher o modelo de linguagem para fine-tuning.
3. Converter os registros para o formato de treinamento apropriado.
4. Realizar o fine-tuning utilizando LoRA ou QLoRA.
5. Comparar os resultados do modelo original com os do modelo personalizado.

O conjunto de teste permanecerá reservado para a avaliação final.
